# DMOSpeech 2: Selective Handover (EXP-020)

**Why.** Teacher-guided sampling passes the teacher's whole estimate to the student. This experiment asks which parts of that estimate the student actually needs. **No model is trained: the released teacher and student are used unchanged.**

| Step | What happens | Passes |
|---|---|---|
| 1 | Teacher runs to step $k$ and gives its estimate $T$ | $2(k+1)$ |
| 2 | One student pass from its own noise gives a draft $S$ | 1 |
| 3 | Split $T$ and $S$ into **voice** (per-band average), **slow** (timing and pitch movement) and **fine** (fast detail), and take each part from the teacher or the student | 0 |
| 4 | The student restarts from the joined result as usual | 3 |

Combinations are named by three letters for (voice, slow, fine): **TTT** is today's handover, **STT** is Voice-Swap (the primary combination), and **SSS** is the student alone.

**Per-input step choice.** Prompt SNR picks a rung for each input: plain Step 4, then STT at Steps 4, 5, 6 or 7, otherwise the reference at Step 8. The cut-offs are calibrated by Learn-Then-Test.

**What runs.** The six mixed combinations at Step 4 (the map) and STT at Steps 5, 6 and 7, on 60 development speakers x 6 prompt conditions (360 cases). Pitch diversity (CVf0) runs on 20 speakers x 20 seeds. Plain Steps 4 to 8 and student-only are reused from EXP-018 and EXP-017.

**Time.** About 5 to 6 hours on a T4 (an estimate, not a measurement). Results are saved to Google Drive after every case, so a disconnected run resumes.

**Status.** Development evidence only. The pass/fail criteria are frozen in `preregistration.json`; the runner refuses to start if it or the preregistration has changed.

---
### Instructions
1. Set the runtime to **T4 GPU** (`Runtime > Change runtime type`).
2. **Runtime > Run all.** The smoke test must pass before the full run can start.
3. If the browser disconnects, reconnect and run **Step 4b**. If the machine resets, re-run Steps 1-2, then Step 4: it continues from the last saved case.

### Step 1: Check the GPU (Tesla T4 required)

In [ ]:
import torch
assert torch.cuda.is_available(), 'CUDA not available. Switch the runtime to T4 GPU.'
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name)
assert 'T4' in gpu_name, f'Expected a Tesla T4 (found {gpu_name}). Student noise depends on the GPU, so the stored labels would not reproduce.'

### Step 2: Clone the Repositories, Install Dependencies, Download Checkpoints and Data

In [ ]:
import os, subprocess, hashlib, sys
THESIS_BRANCH = 'rhnl-3'
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
if not os.path.exists('/content/Final-Thesis-Setup/.git'):
    subprocess.run(['git', 'clone', '-b', THESIS_BRANCH, f"https://{token + '@' if token else ''}github.com/Shonget-Thesis/Final-Thesis-Setup.git", '/content/Final-Thesis-Setup'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Final-Thesis-Setup', 'fetch', 'origin', THESIS_BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/Final-Thesis-Setup', 'checkout', '-B', THESIS_BRANCH, f'origin/{THESIS_BRANCH}'], check=True)

for needed in ('code/colab/run_selective_handover_prelim.py', 'research/experiments/EXP-020-selective-handover-prelim/preregistration.json'):
    if not os.path.exists(f'/content/Final-Thesis-Setup/{needed}'):
        raise FileNotFoundError(f'{needed} is not on branch {THESIS_BRANCH}. Push it to GitHub first.')

DMO_COMMIT = '8a7868fc3345702a586173447f4869205b501f26'
if not os.path.exists('/content/Shonget-DMOSpeech2'):
    !git clone https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git /content/Shonget-DMOSpeech2
%cd /content/Shonget-DMOSpeech2
!git fetch -q origin && git checkout -q {DMO_COMMIT}

!pip install -q "numpy==2.2.3" "pandas==2.2.3" audioop-lts
!pip install -q -r requirements.txt
!pip install -q torchdiffeq vocos speechbrain transformers jiwer soundfile librosa pyarrow scipy jieba accelerate huggingface_hub

!mkdir -p ckpts
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_85000.pt -O ckpts/model_85000.pt
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_1500.pt -O ckpts/model_1500.pt

for fname, digest in {
    'model_85000.pt': 'e143188f7c46b69abb60786937d79b89dac9777d3c4c1e91a97634bcd39ebf2c',
    'model_1500.pt': '6bd50428f3b5f0f25c1a60a0053f40ee47c817bda03c8ee120100f22680ae4c2'
}.items():
    h = hashlib.sha256()
    with open(f'ckpts/{fname}', 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            h.update(block)
    assert h.hexdigest() == digest, f'{fname}: SHA-256 mismatch (delete it and re-run this step)'
    print(f'{fname}: SHA-256 verified')

test_file = 'data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav'
if not os.path.exists(test_file) or os.path.getsize(test_file) < 1000:
    !rm -rf data/seed-tts-eval
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='zhaochenyang20/seed-tts-eval', repo_type='dataset', local_dir='data/seed-tts-eval')
print('Setup and data ready.')

### Step 3: Smoke Test (2 speakers, a few minutes)
Runs every output for 2 development speakers (clean and Gaussian 20 dB) and one short diversity pair, and re-scores plain Steps 4 and 8 through this runner. **If the stored labels do not reproduce, it stops here and the full run will refuse to start.** Writes to a separate `smoke/` folder.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, sys, subprocess
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'{THESIS}/code/colab/run_selective_handover_prelim.py'
PERSIST = '/content/drive/MyDrive/dmospeech2_results'
OUT = f'{PERSIST}/exp020_selective_handover'
LOG = f'{PERSIST}/exp020_selective_handover_log.txt'
os.makedirs(OUT, exist_ok=True)
os.chdir(DMO)
print(f'Results in Google Drive: {OUT}')

proc = subprocess.Popen([sys.executable, '-u', RUNNER, 'smoke', '--out', OUT, '--dmo_root', DMO],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
for line in proc.stdout:
    print(line, end='', flush=True)
assert proc.wait() == 0, f'Smoke test failed with code {proc.returncode}'

import pandas as pd
print(pd.read_csv(f'{OUT}/smoke/selective.csv')[['utt_id', 'condition', 'prompt_snr', 'voice_rms', 'wer_STT_4', 'similarity_STT_4', 'utmos_STT_4', 'wer_TSS_4', 'similarity_TSS_4', 'case_seconds']].to_string())

### Step 4: Full Run + Analysis (about 5 to 6 hours, estimated)
Runs in the background and writes its output to `MyDrive/dmospeech2_results/exp020_selective_handover_log.txt`; this cell prints one progress line per minute. If only the browser disconnects, the job keeps running: reconnect and run **Step 4b**. If the machine resets, re-run Steps 1-2, then this cell.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, sys, subprocess
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'{THESIS}/code/colab/run_selective_handover_prelim.py'
PERSIST = '/content/drive/MyDrive/dmospeech2_results'
OUT = f'{PERSIST}/exp020_selective_handover'
LOG = f'{PERSIST}/exp020_selective_handover_log.txt'
os.makedirs(OUT, exist_ok=True)
os.chdir(DMO)
print(f'Results in Google Drive: {OUT}')

import time, re
cmd = f'{sys.executable} -u "{RUNNER}" run --out "{OUT}" --dmo_root {DMO}'
running = subprocess.run(['pgrep', '-f', 'run_selective_handover_prelim.py'], capture_output=True, text=True).stdout.split()
if running:
    pid = int(running[0])
    alive = lambda: os.path.exists(f'/proc/{pid}')
    print(f'A job is already running (pid {pid}); monitoring it instead of starting a new one.')
else:
    proc = subprocess.Popen(f'nohup {cmd} >> "{LOG}" 2>&1', shell=True)
    alive = lambda: proc.poll() is None
    print(f'Started (pid {proc.pid}). Full log: {LOG}')

last = ''
while alive():
    time.sleep(60)
    try:
        lines = open(LOG, encoding='utf-8', errors='ignore').read().splitlines()
        prog = [l for l in lines if re.match(r'^\[\d+/\d+\]', l) or l.startswith('[OK]') or 'Error' in l or 'SystemExit' in l]
        if prog and prog[-1] != last:
            last = prog[-1]
            print(time.strftime('%H:%M'), last)
    except FileNotFoundError:
        pass
print(f"Finished{'' if running else f' with exit code {proc.returncode}'}. Last lines of the log:")
for line in open(LOG, encoding='utf-8', errors='ignore').read().splitlines()[-15:]:
    print(line)

### Step 4b: Check Progress After a Reconnect
Shows whether the background job is running and how many cases are saved. Does **not** start a second job.

In [ ]:
import os, re, subprocess
PERSIST = '/content/drive/MyDrive/dmospeech2_results'
OUT = f'{PERSIST}/exp020_selective_handover'
LOG = f'{PERSIST}/exp020_selective_handover_log.txt'
running = subprocess.run(['pgrep', '-f', 'run_selective_handover_prelim.py'], capture_output=True, text=True).stdout.split()
print('Job running:', 'yes (pid ' + ', '.join(running) + ')' if running else 'no')
if os.path.exists(LOG):
    lines = open(LOG, encoding='utf-8', errors='ignore').read().splitlines()
    prog = [l for l in lines if re.match(r'^\[\d+/\d+\]', l) or l.startswith('[OK]') or 'Error' in l]
    print('Latest progress:', prog[-1] if prog else '(none yet)')
for name, total in (('selective.csv', 360), ('diversity.csv', 20)):
    path = os.path.join(OUT, name)
    n = sum(1 for _ in open(path, encoding='utf-8')) - 1 if os.path.exists(path) else 0
    print(f'{name}: {n} of {total} saved')

### Step 5: View the Report
If the run stopped early, add `'--allow_partial'` to the command for an interim look (the report is then marked PARTIAL).

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, sys, subprocess
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'{THESIS}/code/colab/run_selective_handover_prelim.py'
PERSIST = '/content/drive/MyDrive/dmospeech2_results'
OUT = f'{PERSIST}/exp020_selective_handover'
LOG = f'{PERSIST}/exp020_selective_handover_log.txt'
os.makedirs(OUT, exist_ok=True)
os.chdir(DMO)
print(f'Results in Google Drive: {OUT}')

from IPython.display import Markdown, display
rep = f'{OUT}/SELECTIVE_HANDOVER_REPORT.md'
if not os.path.exists(rep):
    subprocess.run([sys.executable, RUNNER, 'analyze', '--out', OUT], check=True)
display(Markdown(open(rep, encoding='utf-8').read()))

### Step 6: Zip All Outputs and Download
Packs the CSV files, preregistration, results JSON, report and runner scripts into one ZIP, also saved to Drive. Audio is not included.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, sys, subprocess
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'{THESIS}/code/colab/run_selective_handover_prelim.py'
PERSIST = '/content/drive/MyDrive/dmospeech2_results'
OUT = f'{PERSIST}/exp020_selective_handover'
LOG = f'{PERSIST}/exp020_selective_handover_log.txt'
os.makedirs(OUT, exist_ok=True)
os.chdir(DMO)
print(f'Results in Google Drive: {OUT}')

import zipfile, datetime, torch
zip_name = 'DMOSpeech2_EXP020_Selective_Handover_Results.zip'
git = lambda *a: subprocess.run(['git', *a, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
with open(f'{OUT}/RUN_INFO.txt', 'w') as f:
    f.write(f"dmospeech2_commit: {git()}\n")
    f.write(f"thesis_commit: {git('-C', THESIS)}\n")
    f.write(f"zipped_at_utc: {datetime.datetime.now(datetime.timezone.utc).isoformat()}\n")
    f.write(f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}\n")
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for base, dirs, names in os.walk(OUT):
        for fname in names:
            if fname.startswith('tmp_') or fname.endswith('.wav'):
                continue
            full = os.path.join(base, fname)
            zf.write(full, os.path.relpath(full, OUT))
    for script in ('run_selective_handover_prelim.py', 'run_grid32_prelim.py', 'run_noise_guidance_prelim.py',
                   'run_endpoint_extrapolation_prelim.py', 'run_consistency_gate_prelim.py', 'run_followup_tests.py'):
        zf.write(f'{THESIS}/code/colab/{script}', f'code/{script}')
print(f'Created {zip_name} ({os.path.getsize(zip_name) / 1e6:.2f} MB)')
!cp "{zip_name}" "{PERSIST}/"
print(f'Also saved to Google Drive: {PERSIST}/{zip_name}')
try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print(f'Download manually from the Files panel: {DMO}/{zip_name}')